# 07 - End-to-End Demo: Detect then Classify
Run one image through YOLO detection and classify each detected solar-panel crop.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

DETECTION_ROOT = PROJECT_ROOT / "rooftop-solar-panels-object-detection"
CLASSIFICATION_ROOT = PROJECT_ROOT / "rooftop-solar-panels-image-classification" / "Faulty_solar_panel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DETECTION_ROOT exists:", DETECTION_ROOT.exists())
print("CLASSIFICATION_ROOT exists:", CLASSIFICATION_ROOT.exists())


PROJECT_ROOT: C:\Users\DigvijayYadav\OneDrive - GANIT BUSINESS SOLUTIONS PRIVATE LIMITED\Documents\Object-Identification-Classification-Tutorial
DETECTION_ROOT exists: True
CLASSIFICATION_ROOT exists: True


In [2]:
from pipeline_utils import run_end_to_end_inference
import random

# Auto-resolve latest detection weights
det_candidates = sorted(
    (PROJECT_ROOT / "artifacts/models").glob("det_yolov8n_cpu*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
det_weights = det_candidates[0] if det_candidates else None

# Auto-resolve classification checkpoint
cls_candidates = sorted(
    (PROJECT_ROOT / "artifacts/models").glob("cls_*_cpu.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
cls_ckpt = cls_candidates[0] if cls_candidates else None

# Strictly pick sample image from classification TEST split
RANDOM_TEST_SAMPLE = True
SEED = 42

classification_index = PROJECT_ROOT / "data/processed/classification_index.csv"
if not classification_index.exists():
    raise FileNotFoundError("Missing data/processed/classification_index.csv. Run notebook 05 first.")

split_df = pd.read_csv(classification_index)
required_cols = {"file_path", "split", "label"}
if not required_cols.issubset(split_df.columns):
    raise ValueError(f"classification_index.csv must contain columns: {sorted(required_cols)}")

def resolve_test_path(row):
    p = Path(str(row["file_path"]))
    if p.exists():
        return p
    alt = CLASSIFICATION_ROOT / str(row["label"]) / p.name
    if alt.exists():
        return alt
    return None

test_df = split_df[split_df["split"] == "test"].copy()
test_df["resolved_path"] = test_df.apply(resolve_test_path, axis=1)
test_df = test_df[test_df["resolved_path"].notna()]
if test_df.empty:
    raise ValueError("No valid existing files found in classification test split after path resolution.")

if RANDOM_TEST_SAMPLE:
    sampled_row = test_df.sample(n=1, random_state=SEED).iloc[0]
else:
    sampled_row = test_df.sort_values("resolved_path").iloc[0]

sample_image = Path(sampled_row["resolved_path"])
actual_class = str(sampled_row["label"])

print("Resolved sample_image (test split):", sample_image)
print("Actual class:", actual_class)
print("Resolved det_weights:", det_weights)
print("Resolved cls_ckpt:", cls_ckpt)
print("Random test sample:", RANDOM_TEST_SAMPLE)

if det_weights is None:
    raise FileNotFoundError("No detection best.pt found under artifacts/models/det_yolov8n_cpu*/weights/")
if cls_ckpt is None:
    raise FileNotFoundError("No classification checkpoint found under artifacts/models/cls_*_cpu.pt")

out_df = run_end_to_end_inference(
    sample_image=sample_image,
    det_weights=det_weights,
    cls_checkpoint=cls_ckpt,
    conf=0.25,
    output_dir=PROJECT_ROOT / "artifacts/predictions/end_to_end",
    actual_class=actual_class,
)

display(out_df)

if not out_df.empty and "annotated_image_path" in out_df.columns:
    print("Annotated image:", out_df.loc[0, "annotated_image_path"])
    print("Run folder:", out_df.loc[0, "run_dir"])



Resolved sample_image (test split): C:\Users\DigvijayYadav\OneDrive - GANIT BUSINESS SOLUTIONS PRIVATE LIMITED\Documents\Object-Identification-Classification-Tutorial\rooftop-solar-panels-image-classification\Faulty_solar_panel\Bird-drop\Bird (43).jpg
Actual class: Bird-drop
Resolved det_weights: C:\Users\DigvijayYadav\OneDrive - GANIT BUSINESS SOLUTIONS PRIVATE LIMITED\Documents\Object-Identification-Classification-Tutorial\artifacts\models\det_yolov8n_cpu3\weights\best.pt
Resolved cls_ckpt: C:\Users\DigvijayYadav\OneDrive - GANIT BUSINESS SOLUTIONS PRIVATE LIMITED\Documents\Object-Identification-Classification-Tutorial\artifacts\models\cls_mobilenetv3_small_cpu.pt
Random test sample: True


,image,bbox_id,det_confidence,actual_class,predicted_condition,condition_probability,crop_path,annotated_image_path,run_dir
0,C:\Users\DigvijayYadav\OneDrive - GANIT BUSINE...,0,0.869052,Bird-drop,Bird-drop,0.999848,C:\Users\DigvijayYadav\OneDrive - GANIT BUSINE...,C:\Users\DigvijayYadav\OneDrive - GANIT BUSINE...,C:\Users\DigvijayYadav\OneDrive - GANIT BUSINE...


Annotated image: C:\Users\DigvijayYadav\OneDrive - GANIT BUSINESS SOLUTIONS PRIVATE LIMITED\Documents\Object-Identification-Classification-Tutorial\artifacts\predictions\end_to_end\Bird (43)_20260303_151659\Bird (43)_annotated.jpg
Run folder: C:\Users\DigvijayYadav\OneDrive - GANIT BUSINESS SOLUTIONS PRIVATE LIMITED\Documents\Object-Identification-Classification-Tutorial\artifacts\predictions\end_to_end\Bird (43)_20260303_151659
